In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_dir = project_root / "data" / "05_bond_book_yield_curve_twist"
if not data_dir.is_dir():
    raise FileNotFoundError("Open this notebook from the repository root or notebooks directory.")

positions = pd.read_csv(data_dir / "positions.csv")
curve = pd.read_csv(data_dir / "curve.csv")
cashflows = pd.read_csv(data_dir / "cashflows.csv")

print(positions)
print(curve)
print(cashflows)


  bond_id  signed_units
0     B03           -50
1     B01            80
2     B02          -120
   maturity_years  zero_rate_pct  shock_bps
0             0.5           2.30         30
1             1.0           2.50         20
2             2.0           2.75        -10
3             3.0           3.00        -35
  bond_id  payment_years  cashflow_eur
0     B01            1.0         100.0
1     B02            0.5           2.0
2     B02            1.0           2.0
3     B02            1.5         102.0
4     B03            0.5           2.5
5     B03            1.0           2.5
6     B03            1.5           2.5
7     B03            2.0           2.5
8     B03            2.5         102.5


In [2]:
def bond_prices(coupons, zero_rates, payment_years_sorted):
    P = sum(coupons.iloc[i] * np.exp(-zero_rates.iloc[i] * payment_years_sorted.iloc[i]) for i in range(len(payment_years_sorted)))
    return P

def linear_inter(lower_time, lower_value, upper_time, upper_value, middle_time):
    rate = (np.abs(middle_time - lower_time) / (upper_time - lower_time)) * upper_value + (np.abs(upper_time - middle_time) / (upper_time - lower_time)) * lower_value
    return rate

In [6]:

# Renames the column `maturity_years` to `payment_years` so curve can be joined to cash flows
curve_rename = curve.rename(columns={"maturity_years": "payment_years"})

cashflows_new = cashflows.merge(curve_rename,
                               how="left",
                               on="payment_years")



# Data missing from cash-flows, we identify which payment years are not present in `curve` and store it in `missing_years` by using `.isna()` mask
bond_payment_years_not_in_curve = cashflows_new[cashflows_new["zero_rate_pct"].isna()].reset_index(drop=True)

missing_years = bond_payment_years_not_in_curve["payment_years"].drop_duplicates()


# Calculates the distance between each `missing_years` and all the `payment_years`
distances = [
    curve["maturity_years"].sub(year).abs() for year in missing_years
]





#For each missing date, interpolate between the nearest quoted maturities on either side.

new_rate = pd.Series(dtype="float64", name="new_zero_rate_pct")
new_shock = pd.Series(dtype="float64", name="new_shock_bps")

for year, distances_for_year in zip(missing_years, distances):

    # Creates first boolean condition as a mask to filter all the years above and below our `missing_years`
    above = year < curve["maturity_years"]
    below = year > curve["maturity_years"]

    # Given the above mask we filter the values that have the smallest `distances` in their maturity
    row_above = curve.loc[distances_for_year[above].idxmin()]
    row_below = curve.loc[distances_for_year[below].idxmin()]

    # Calculate interpolated values and concat
    zero_rate_inter = linear_inter(row_below["maturity_years"], row_below["zero_rate_pct"], row_above["maturity_years"], row_above["zero_rate_pct"], year)

    shock_bps_inter = linear_inter(row_below["maturity_years"], row_below["shock_bps"], row_above["maturity_years"], row_above["shock_bps"], year)

    new_rate_rows = pd.Series([zero_rate_inter])
    new_rate = pd.concat([new_rate, new_rate_rows], ignore_index=True)

    new_shock_rows = pd.Series([shock_bps_inter])
    new_shock = pd.concat([new_shock, new_shock_rows], ignore_index=True)

# Concatinate the `pd.Series` into a dataframe
new_rate_n_shock = pd.concat([missing_years.reset_index(drop=True), new_rate, new_shock],
                              axis=1, keys=["payment_years", "new_zero_rate_pct", "new_shock_bps"])

# Merge everything and fill in the missing values & discard unecessary collumns
cashflows_complete = cashflows_new.merge(new_rate_n_shock,
                                     how="left",
                                     on="payment_years")


cashflows_complete["zero_rate_pct"] = cashflows_complete["zero_rate_pct"].fillna(cashflows_complete["new_zero_rate_pct"])
cashflows_complete["shock_bps"] = cashflows_complete["shock_bps"].fillna(cashflows_complete["new_shock_bps"])

cashflows_complete = cashflows_complete.iloc[:, 0:5]





# Creates an empty dictionary to store results
results = {}

# Writes a mask for each `bond_id`-word and stores it in the `results` dictionary
for i in range(len(cashflows_complete["bond_id"].drop_duplicates().reset_index(drop=True))):
    new_results = cashflows_complete["bond_id"] == cashflows_complete["bond_id"].drop_duplicates().reset_index(drop=True).iloc[i]

    results[cashflows_complete["bond_id"].drop_duplicates().reset_index(drop=True).iloc[i]] = new_results
    




# Creates empty list to populate with dictionaries for each of `bond_id` and their respective `price`
rows = []
for id in cashflows_complete["bond_id"]:

    row = {"bond_id": id, 
           "price": bond_prices(cashflows_complete["cashflow_eur"].loc[results[id]], cashflows_complete["zero_rate_pct"].loc[results[id]] / 100, cashflows_complete["payment_years"].loc[results[id]])}
    rows.append(row)

# Finally creates a dataframe with bonds and their prices
bond_prices_df = pd.DataFrame(rows).drop_duplicates()

print(bond_prices_df)


  bond_id       price
0     B01   97.530991
1     B02  101.989544
4     B03  105.070718


In [4]:
payment_years_sorted = cashflows["payment_years"].sort_values(ascending=True).drop_duplicates()

print(len(payment_years_sorted))

5
